# 05 — Combinar autores UNAM

Une los autores UNAM identificados automáticamente con los casos corregidos manualmente.

El resultado se guarda en `../04_Limpieza/02_normalizacion/autores_unam_completos.csv`.

In [1]:
import os
import pandas as pd

archivo_automatico = "../04_Limpieza/01_Internos_unam/autores_unam_identificados.csv"
archivo_manual = "../04_Limpieza/00_Separacion_Autor_Afiliacion/articulos_revision_completa.csv"
carpeta_salida = "../04_Limpieza/02_normalizacion"
archivo_salida = f"{carpeta_salida}/autores_unam_completos.csv"

os.makedirs(carpeta_salida, exist_ok=True)

columnas = [
    "Base_origen", "Fuente_origen", "indice", "Titulo", "Año",
    "Autor_norm", "Afiliacion1", "Afiliacion2", "ISBN", "ISSN",
    "Doi", "URL", "Area", "SubArea", "Keywords", "Abstract"
]

automatico = pd.read_csv(
    archivo_automatico,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

manual = pd.read_csv(
    archivo_manual,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

# Conservar solo las 16 columnas canónicas
automatico = automatico[columnas].copy()
manual = manual[columnas].copy()

# Eliminar únicamente filas completamente vacías del archivo manual
manual = manual[manual.apply(lambda fila: fila.str.strip().ne("").any(), axis=1)].copy()

# Unir sin deduplicar ni modificar valores
completo = pd.concat([automatico, manual], ignore_index=True)

# Validaciones simples
if list(completo.columns) != columnas:
    raise ValueError("La salida no conserva las 16 columnas canónicas.")

if completo[["Base_origen", "indice", "Autor_norm"]].duplicated().any():
    raise ValueError("Se encontraron autores duplicados para la misma fila original.")

completo.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("Filas automáticas:", len(automatico))
print("Filas manuales:", len(manual))
print("Filas totales:", len(completo))
print("Columnas:", len(completo.columns))
print("Guardado en:", archivo_salida)

Filas automáticas: 4152
Filas manuales: 114
Filas totales: 4266
Columnas: 16
Guardado en: ../04_Limpieza/02_normalizacion/autores_unam_completos.csv
